In [6]:
#importacion

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from pathlib import Path
from tqdm import tqdm

In [7]:
# Rutas

ruta_carpeta = Path(
    "../Resultados/01_Tablas_de_resultados"
)

ruta_salida = Path(
    "../Resultados/02_Tabla_de_metricas_por_mes"
)
ruta_salida.mkdir(parents=True, exist_ok=True)

Bloque de definición de métricas deterministicas que comparan el promedio de los ensambles con el simulado con clima histórico:

In [8]:
def nash_sutcliffe(q_obs, q_sim):
    return 1 - np.sum((q_obs - q_sim)**2) / np.sum((q_obs - np.mean(q_obs))**2)

def pbias(q_obs, q_sim):
    return 100 * np.sum(q_sim - q_obs) / np.sum(q_obs)

def cv_rmse(q_obs, q_sim):
    rmse = np.sqrt(np.mean((q_sim - q_obs)**2))
    return rmse / np.mean(q_obs)

def pearson_corr(q_obs, q_sim):
    return pearsonr(q_obs, q_sim)[0]

Definición del calculo del crpss_ad (métrica probabilistica)

In [9]:
#esta es la función para calcular CRPS para una distribución, tiene dos terminos: el primer evalúa la diferencia entre el promedio de 
#los ensambles con el valor observado, el segundo evalúa la dispersión del ensamble analizando la diferencia entre todos los membros.
def crps(ens, obs):
    # primero ordenamos los ensambles de menor a mayor
    ens = np.sort(ens)
    #calcula el promedio de la distancia entre cada ensamble y el observado
    term1 = np.mean(np.abs(ens - obs))
    #acá vemos a crear un lista de "columnas" y una lista de "filas" y vamos crear una matriz con la diferencia entre cada columna y fila
    #obtenemos la media de esta matriz y así medimos la dispersión entre los ensambles
    term2 = 0.5 * np.mean(np.abs(ens[:, None] - ens[None, :]))
    #este es el valor de crps
    return term1 - term2

#acá se calcula el CRPS para el ESP (sería como el ESP performa para cada enero de cada año, despues se promedia el valor de la métrica
#entre todos los eneros) y también el CRPS para el ensamble de referencia que sería la climatología (como la climatología se desenpeña
#para predecir el valor de cada enero)
def crpss_ad(df_sub, ens_cols):
    #clonar
    df_sub = df_sub.copy()
    
    climatologia = df_sub["QC_hist"].values

    # Listas vacías para ir guardando 
    crps_model = []
    crps_ref = []

    #Empieza a recorrer el DataFrame fila por fila.
    for _, row in df_sub.iterrows():
        #Toma qc_hist de esa fila.
        obs = row["QC_hist"]
        #Si no hay dato, salta esta fila.
        if np.isnan(obs):
            continue
            
        # --- Ensamble del modelo ---
        #Toma solo las columnas que empiezan con "ens_", sacando ens_prom
        ens = row[ens_cols].dropna().values

        # --- Ensamble climatológico (referencia para avaliar) ---
        #crada anteriormente.
        clim_ens = climatologia
            
        # quita el valor observado actual de la lista histórica
        clim_ens = clim_ens[clim_ens != obs]
        if len(clim_ens) < 2:
            continue

        # --- aplicar el CRPS a los ensambles del modelo y de referencia ---
        crps_model.append(crps(ens, obs))
        crps_ref.append(crps(clim_ens, obs))

    return 1 - np.mean(crps_model) / np.mean(crps_ref)

Definición del cálculo del brier escore para diferentes percentiles

In [10]:
def brier_score_q10(df_sub, ens_cols, q10):
    # toma el df organizado por leadtime y mes.
    df_sub = df_sub.copy()

    #contenedor para resultados
    bs = []

    for _, row in df_sub.iterrows():
        #para cada fila obtener los valores de los ensambles
        ens = row[ens_cols].dropna().values
        #para cada fila obtener el valor simluado histórico
        obs = row["QC_hist"]

        # filtro por si algun mes no tiene ensambles suficientes o simulado histórico
        if len(ens) < 2 or np.isnan(obs):
            continue

        # empezamos a calcular cuantos ensambles caen en el umbral
        p = np.mean(ens < q10)

        #el observado, cae dentro del umbral?
        o = int(obs < q10)

        # guardamos el error cuadrático del pronóstico para este mes específico.
        bs.append((p - o) ** 2)
                
    # el brier score final es el promedio de todos esos errores cuadráticos mensuales.
    return np.mean(bs)

def brier_score_q10_q25(df_sub, ens_cols, q10, q25):
    # toma el df organizado por leadtime y mes.
    df_sub = df_sub.copy()

    #contenedor para resultados
    bs = []

    for _, row in df_sub.iterrows():
        ens = row[ens_cols].dropna().values
        obs = row["QC_hist"]

        if len(ens) < 2 or np.isnan(obs):
            continue

        p = np.mean((ens >= q10) & (ens < q25))

        o = int((obs >= q10) & (obs < q25))

        bs.append((p - o) ** 2)

    return np.mean(bs)


def brier_score_q25_q75(df_sub, ens_cols, q25, q75):
    # toma el df organizado por leadtime y mes.
    df_sub = df_sub.copy()

    #contenedor para resultados
    bs = []

    for _, row in df_sub.iterrows():
        ens = row[ens_cols].dropna().values
        obs = row["QC_hist"]

        if len(ens) < 2 or np.isnan(obs):
            continue

        p = np.mean((ens >= q25) & (ens <= q75))

        o = int((obs >= q25) & (obs <= q75))

        bs.append((p - o) ** 2)

    return np.mean(bs)

    
def brier_score_q75_q90(df_sub, ens_cols, q75, q90):
    # toma el df organizado por leadtime y mes.
    df_sub = df_sub.copy()

    #contenedor para resultados
    bs = []

    for _, row in df_sub.iterrows():
        ens = row[ens_cols].dropna().values
        obs = row["QC_hist"]


        if len(ens) < 2 or np.isnan(obs):
            continue

        p = np.mean((ens > q75) & (ens <= q90))

        o = int((obs > q75) & (obs <= q90))

        bs.append((p - o) ** 2)

    return np.mean(bs)



def brier_score_q90(df_sub, ens_cols, q90):
    df_sub = df_sub.copy()
    bs = []

    for _, row in df_sub.iterrows():
        ens = row[ens_cols].dropna().values
        obs = row["QC_hist"]

        if len(ens) < 2 or np.isnan(obs):
            continue

        p = np.mean(ens >= q90)
                    
        o = int(obs > q90)

        bs.append((p - o) ** 2)

    return np.mean(bs)


Brier climatologia

In [11]:
def brier_score_climatologia_q10(df_sub, q10):
    bs = []

    # probabilidad climatológica
    p_clim = 0.10

    for _, row in df_sub.iterrows():

        obs = row["QC_hist"]

        if np.isnan(obs):
            continue

        # observación binaria
        o = int(obs < q10)

        bs.append((p_clim - o) ** 2)

    return np.mean(bs)

    
def brier_score_climatologia_q10_q25(df_sub, q10, q25):
    bs = []

    # probabilidad climatológica fija
    p_clim = 0.15

    for _, row in df_sub.iterrows():

        obs = row["QC_hist"]

        if np.isnan(obs):
            continue

        o = int((obs >= q10) & (obs < q25))

        bs.append((p_clim - o) ** 2)

    return np.mean(bs)

def brier_score_climatologia_q25_q75(df_sub, q25, q75):
    bs = []

    # probabilidad climatológica fija
    p_clim = 0.5

    for _, row in df_sub.iterrows():

        obs = row["QC_hist"]

        if np.isnan(obs):
            continue

        o = int((obs >= q25) & (obs < q75))

        bs.append((p_clim - o) ** 2)

    return np.mean(bs)

def brier_score_climatologia_q75_q90(df_sub, q75, q90):
    bs = []

    # probabilidad climatológica fija
    p_clim = 0.15

    for _, row in df_sub.iterrows():

        obs = row["QC_hist"]

        if np.isnan(obs):
            continue

        o = int((obs >= q75) & (obs < q90))

        bs.append((p_clim - o) ** 2)

    return np.mean(bs)


def brier_score_climatologia_q90(df_sub, q90):
    bs = []
    # probabilidad climatológica
    p_clim = 0.10

    for _, row in df_sub.iterrows():

        obs = row["QC_hist"]
        
        if np.isnan(obs):
            continue

        # observación binaria
        o = int(obs > q90)

        bs.append((p_clim - o) ** 2)

    return np.mean(bs)


Calculo BrierScore

In [12]:
def brier_skill_score(bs_modelo, bs_clim):

    return 1 - (bs_modelo / bs_clim)

Loop sobre las tablas de resultados para calcular las métricas:

In [15]:
archivos = list(ruta_carpeta.glob("*.csv"))
for ruta_csv in tqdm(archivos, desc="Procesando cuencas"):
    
    df = pd.read_csv(ruta_csv)

    df["fecha_emision"] = pd.to_datetime(df["fecha_emision"])
    df["fecha_pronostico"] = pd.to_datetime(df["fecha_pronostico"])

    df["mes"] = df["fecha_pronostico"].dt.month

    ens_cols = [c for c in df.columns if c.startswith("ens_") and c != "ens_prom"]

    resultados = []

    for (lead, mes), df_sub in df.groupby(["lead", "mes"]):

        q_obs = df_sub["QC_hist"].values
        q_sim = df_sub["ens_prom"].values

        q10 = np.percentile(q_obs[~np.isnan(q_obs)], 10)
        q25 = np.percentile(q_obs[~np.isnan(q_obs)], 25)
        q75 = np.percentile(q_obs[~np.isnan(q_obs)], 75)
        q90 = np.percentile(q_obs[~np.isnan(q_obs)], 90)

        # -----------------------------
        # BRIER SCORES
        # -----------------------------

        bs_q10 = brier_score_q10(df_sub, ens_cols, q10)
        bs_clim_q10 = brier_score_climatologia_q10(df_sub, q10)

        bs_q10_q25 = brier_score_q10_q25(df_sub, ens_cols, q10, q25)
        bs_clim_q10_q25 = brier_score_climatologia_q10_q25(df_sub, q10, q25)

        bs_q25_q75 = brier_score_q25_q75(df_sub, ens_cols, q25, q75)
        bs_clim_q25_q75 = brier_score_climatologia_q25_q75(df_sub, q25, q75)

        bs_q75_q90 = brier_score_q75_q90(df_sub, ens_cols, q75, q90)
        bs_clim_q75_q90 = brier_score_climatologia_q75_q90(df_sub, q75, q90)

        bs_q90 = brier_score_q90(df_sub, ens_cols, q90)
        bs_clim_q90 = brier_score_climatologia_q90(df_sub, q90)

        # -----------------------------
        # BRIER SKILL SCORES
        # -----------------------------

        bss_q10 = brier_skill_score(bs_q10, bs_clim_q10)
        bss_q10_q25 = brier_skill_score(bs_q10_q25, bs_clim_q10_q25)
        bss_q25_q75 = brier_skill_score(bs_q25_q75, bs_clim_q25_q75)
        bss_q75_q90 = brier_skill_score(bs_q75_q90, bs_clim_q75_q90)
        bss_q90 = brier_skill_score(bs_q90, bs_clim_q90)

        metricas = {

            "lead": lead,
            "mes": mes,
            "NS": nash_sutcliffe(q_obs, q_sim),
            "Pbias": pbias(q_obs, q_sim),
            "CV": cv_rmse(q_obs, q_sim),
            "Pearson": pearson_corr(q_obs, q_sim),
            "CRPSSad": crpss_ad(df_sub, ens_cols),

            # -----------------------------
            # BRIER SKILL SCORE
            # -----------------------------

            "Brier_Q10": bss_q10,
            "Brier_Q10_Q25": bss_q10_q25,
            "Brier_Q25_Q75": bss_q25_q75,
            "Brier_Q75_Q90": bss_q75_q90,
            "Brier_Q90": bss_q90,

            "N_det": np.sum(~np.isnan(q_sim)),
            "N_prob": np.sum(df_sub[ens_cols].notna().sum(axis=1) >= 2)
        }
        
        resultados.append(metricas)

    df_out = pd.DataFrame(resultados)

    df_out.to_csv(
        ruta_salida / f"{ruta_csv.stem}_metricas_por_mes.csv",
        index=False
    )

Procesando cuencas: 100%|██████████████████████████████████████████████████████████████| 47/47 [11:04<00:00, 14.13s/it]
